# Tutorial 01 — Model 1: Single Expanding Conductive Plume

This notebook walks through every stage of the HWSI workflow for the single-plume scenario:

1. Mesh construction (forward and inversion meshes)
2. True resistivity model definition
3. Synthetic data generation
4. Running all four inversion methods
5. Error evaluation (nRMSE)
6. Figure generation

Each cell is self-contained and annotated.  You can run cells individually to follow the workflow step by step.

---

**Dependencies**: pyGIMLi 1.5.5, NumPy ≥ 1.23, SciPy ≥ 1.9, Matplotlib ≥ 3.6  
See `README.md` for installation instructions.

## 0. Imports

In [ ]:
import sys, os
# Make sure the repository root is on the path when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import Normalize

import pygimli.meshtools as mt
from pygimli.physics import ert

from hwsi import (
    HWSI_Inversion,
    run_independent,
    run_difference,
    run_4d,
    interp_to_grid,
    build_valid_mask,
    calc_errors,
)

%matplotlib inline
print('All imports successful.')

## 1. Mesh Construction

Two meshes with deliberately different topology are used to avoid the **inverse crime**: using the same discretisation for both data generation and inversion would make the reconstruction artificially easy.

- **Forward mesh**: large domain (±25 m × 15 m), coarser resolution (area ≤ 0.2 m²)
- **Inversion mesh**: restricted domain (±12 m × 6 m), finer resolution (area ≤ 0.1 m²)

In [ ]:
# Dipole-dipole electrode layout: 50 electrodes over 24 m
scheme = ert.createData(elecs=np.linspace(-12, 12, 50), schemeName='dd')

# --- Forward mesh ---
world = mt.createWorld(start=[-25, -15], end=[25, 0], worldMarker=True)
for p in scheme.sensorPositions():
    world.createNode(p)
fwd_mesh = mt.createMesh(world, quality=34.5, area=0.2)

# --- Inversion mesh ---
inv_domain = mt.createRectangle(start=[-12, -6], end=[12, 0])
inv_mesh   = mt.createMesh(inv_domain, quality=34.5, area=0.1)

# Regular grid for visualisation and error evaluation
grid_x, grid_z = np.mgrid[-12:12:300j, -6:0:150j]

print(f'Forward mesh : {fwd_mesh.cellCount():,} cells')
print(f'Inversion mesh: {inv_mesh.cellCount():,} cells')

## 2. True Resistivity Model

A single conductive plume (10 Ωm) expands semicircularly from the surface into a 600 Ωm background.  The plume radius grows linearly from 0.5 m at t = 0 h to 4.0 m at t = 24 h, giving a 60:1 contrast ratio.

In [ ]:
BG_RHO    = 600.0  # background resistivity (Ωm)
PLUME_RHO = 10.0   # plume resistivity (Ωm)
R_MIN, R_MAX = 0.5, 4.0   # radius range (m)

def get_true_rho_grid(gx, gz, t_hour):
    """True resistivity on the regular grid at time t_hour."""
    rho = np.full(gx.shape, BG_RHO)
    if t_hour <= 0:
        return rho
    rad  = R_MIN + (R_MAX - R_MIN) * (t_hour / 24.0)
    dist = np.sqrt(gx**2 + gz**2)  # centred at surface origin
    rho[(dist <= rad) & (gz <= 0)] = PLUME_RHO
    return rho

def get_true_rho_point(x, z, t_hour):
    """True resistivity at a single point and time."""
    if z > 0 or t_hour <= 0:
        return BG_RHO
    rad = R_MIN + (R_MAX - R_MIN) * (t_hour / 24.0)
    return PLUME_RHO if np.sqrt(x**2 + z**2) <= rad else BG_RHO

# Quick visualisation of the true model at t = 12 h
tg_12 = get_true_rho_grid(grid_x, grid_z, 12)
fig, ax = plt.subplots(figsize=(8, 3))
im = ax.imshow(
    np.log10(tg_12).T, origin='lower', cmap='jet',
    vmin=0, vmax=4, extent=[-12, 12, -6, 0], aspect='equal',
)
plt.colorbar(im, ax=ax, label='log10(rho) [Ωm]')
ax.set_title('True model at t = 12 h')
ax.set_xlabel('Distance (m)'); ax.set_ylabel('Depth (m)')
plt.tight_layout()
plt.show()

## 3. Synthetic Data Generation

Seven epochs are simulated at 4-hour intervals (t = 0, 4, 8, 12, 16, 20, 24 h).  Each dataset is contaminated with 3% Gaussian noise; a conservative 5% data error is assumed in inversion.

In [ ]:
NOISE_LEVEL = 0.03
DATA_ERROR  = 0.05
times = np.arange(0, 25, 4)   # [0, 4, 8, 12, 16, 20, 24]

data_list, true_grids = [], []

print('Generating synthetic datasets...')
for t in times:
    rho_vec = np.array([
        get_true_rho_point(c.center().x(), c.center().y(), t)
        for c in fwd_mesh.cells()
    ])
    data = ert.simulate(
        mesh=fwd_mesh, scheme=scheme, res=rho_vec,
        noiseLevel=NOISE_LEVEL, seed=123, verbose=False,
    )
    data['rhoa'] = np.maximum(np.abs(data['rhoa']), 0.1)
    data.set('err', np.full(data.size(), DATA_ERROR))
    data_list.append(data)

    tg = get_true_rho_grid(grid_x, grid_z, t)
    true_grids.append(tg)
    r_now = R_MIN + (R_MAX - R_MIN) * (t / 24.0) if t > 0 else 0.0
    print(f'  T={t:2d}h | plume cells: {int(np.sum(tg < 50)):5d} | r={r_now:.2f} m')

print('Done.')

## 4. Running the Inversions

All four methods use the same spatial regularisation strength (λ_s = 20).  HWSI additionally uses λ_t = 20, ε = 5×10⁻³, α_max = 0.65.

In [ ]:
LAMBDA_S = 20.0
LAMBDA_T = 20.0

# Independent inversion — no temporal coupling
results_indep = run_independent(inv_mesh, data_list, lam=LAMBDA_S)

In [ ]:
# Difference inversion — change relative to baseline epoch
results_diff = run_difference(inv_mesh, data_list, lam=LAMBDA_S)

In [ ]:
# 4D L2-coupled inversion — uniform temporal penalty
results_4d = run_4d(inv_mesh, data_list, lam=LAMBDA_S, scalef=1.0)

In [ ]:
# HWSI — adaptive per-cell temporal coupling
hwsi = HWSI_Inversion(
    inv_mesh, data_list,
    lambda_s=LAMBDA_S, lambda_t=LAMBDA_T,
    epsilon=5e-3, alpha_max=0.65,
    max_iter=15, verbose=True,
)
results_hwsi = hwsi.run()

## 5. Error Evaluation

In [ ]:
results_dict = {
    'Independent':   results_indep,
    'Difference':    results_diff,
    '4D L2-Coupled': results_4d,
    'HWSI':          results_hwsi,
}

valid_mask = build_valid_mask(inv_mesh, grid_x, grid_z)
errors     = calc_errors(
    times, true_grids, results_dict,
    grid_x, grid_z, inv_mesh, mask=valid_mask,
)

baseline = np.nanmean(errors['Independent']['nrmse'])
print(f"{'Method':<20} {'Mean nRMSE':>11} {'Std':>7} {'vs Indep':>10}")
print('-' * 52)
for mn in results_dict:
    avg = np.nanmean(errors[mn]['nrmse'][1:])
    std = np.nanstd(errors[mn]['nrmse'][1:])
    print(f"{mn:<20} {avg:>10.2f}% {std:>6.2f}%  {baseline - avg:>+8.2f} pp")

## 6. Quick nRMSE Plot

In [ ]:
METHOD_COLORS = {
    'Independent':   '#1f77b4',
    'Difference':    '#ff7f0e',
    '4D L2-Coupled': '#d62728',
    'HWSI':          '#2ca02c',
}

fig, ax = plt.subplots(figsize=(9, 5))
xt = range(1, len(times))
for mn, vals in errors.items():
    ax.plot(xt, vals['nrmse'][1:], lw=2.2,
            label=mn, color=METHOD_COLORS[mn])
ax.set_xticks(list(xt))
ax.set_xticklabels([f'T={t}h' for t in times[1:]])
ax.set_ylabel('nRMSE (%)')
ax.set_ylim(20, 100)
ax.grid(alpha=0.3, ls='--')
ax.legend()
plt.tight_layout()
plt.show()

## 7. Inversion Model Comparison at t = 20 h

The t = 20 h panel is where the methods diverge most strongly.

In [ ]:
ti_20 = list(times).index(20)
norm  = Normalize(vmin=0, vmax=4)

fig, axes = plt.subplots(1, 4, figsize=(20, 4))
for ax, (mn, res) in zip(axes, results_dict.items()):
    gd = interp_to_grid(inv_mesh, res[ti_20], grid_x, grid_z)
    ax.imshow(
        np.log10(np.clip(gd, 1, 1e4)).T,
        origin='lower', cmap='jet', norm=norm,
        extent=[-12, 12, -6, 0], aspect='equal',
    )
    ax.set_title(f'{mn}\nnRMSE = {errors[mn]["nrmse"][ti_20]:.1f}%')
    ax.set_xlabel('Distance (m)')
axes[0].set_ylabel('Depth (m)')
plt.suptitle('Inverted resistivity at t = 20 h', fontsize=13, y=1.02)
plt.tight_layout()
plt.show()

## 8. HWSI Convergence

In [ ]:
hist  = hwsi.convergence_history
iters = [h['iteration']  for h in hist]
avgs  = [h['avg_change'] for h in hist]

fig, ax = plt.subplots(figsize=(5, 4))
ax.semilogy(iters, avgs, color='#2ca02c', lw=2.4)
ax.set_xlabel('Iteration')
ax.set_ylabel(r'Avg $||\Delta m||$')
ax.set_xticks(iters)
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()
print(f'Declared convergence at iteration {iters[-1]}')

---

For high-resolution publication figures, run `models/model1_single_plume.py` directly.  
For the two-plume scenario, open `02_model2_tutorial.ipynb`.